# F4
Lucas Andersen (pbp8nq@virginia.edu​) 

DS 5001

May 8, 2026

In [58]:
import pandas as pd
import numpy as np
import os

In [59]:
F1_path = 'data/F1'
F3_path = 'data/F3'
F4_path = 'data/F4'

os.makedirs(F4_path, exist_ok = True)

In [60]:
OHCO = ['work_id','part_num','chap_num','para_num','sent_num','token_num']
WORKS = OHCO[:1]
PARAS = OHCO[:4]

## Load F3 Data

In [61]:
LIB = pd.read_csv(f"{F1_path}/LIB.csv", index_col = OHCO[0])
TOKEN = pd.read_csv(f"{F3_path}/TOKEN.csv", index_col = OHCO, keep_default_na=False, na_values=[])
VOCAB = pd.read_csv(f"{F3_path}/VOCAB.csv", index_col = 'term_id', keep_default_na = False, na_values=[])

print(f'LIB: {len(LIB)} works')
print(f"TOKEN: {len(TOKEN):,} tokens")
print(f"VOCAB: {len(VOCAB):,} terms")

LIB: 49 works
TOKEN: 284,077 tokens
VOCAB: 15,394 terms


## Drop empty `term_str` rows

In [62]:
VOCAB = VOCAB[VOCAB.term_str != '']
TOKEN = TOKEN[TOKEN.term_str != '']

print(f"TOKEN: {len(TOKEN):,} tokens")
print(f"VOCAB: {len(VOCAB):,} terms")

TOKEN: 283,960 tokens
VOCAB: 15,393 terms


## Add `term_id` to TOKEN

In [63]:
TOKEN['term_id'] = TOKEN.term_str.map(VOCAB.reset_index().set_index('term_str').term_id)

## Add statistical features to VOCAB

#### `term_rank`
Rank by frequency (1 = most common)

In [64]:
if 'term_rank' not in VOCAB.columns:
    VOCAB = VOCAB.sort_values('n', ascending=False).reset_index()
    VOCAB.index.name = 'term_rank'
    VOCAB = VOCAB.reset_index()
    VOCAB = VOCAB.set_index('term_id')
    VOCAB['term_rank'] = VOCAB['term_rank'] + 1

VOCAB.head()

,term_rank,term_str,n,num,stop,p_stem,pos_max,wn_pos,lemma
term_id,,,,,,,,,
13682,1,the,14903,0,1,the,DT,n,the
9358,2,of,10196,0,1,of,IN,n,of
726,3,and,9565,0,1,and,CC,n,and
13874,4,to,9088,0,1,to,TO,n,to
202,5,a,6707,0,1,a,DT,n,a


#### `p` and `zipf_k`
Probability and Zipf's K

In [65]:
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['zipf_k'] = VOCAB.n * VOCAB.term_rank

#### `h`
Entropy

In [66]:
VOCAB['h'] = VOCAB.p * np.log2(1/VOCAB.p) # self entropy of each word

H = VOCAB.h.sum()
N_v = VOCAB.shape[0]
H_max = np.log2(N_v)
R = round(1 - (H/H_max), 2) * 100

print("H \t= {}\nH_max \t= {}\nR \t= {}%".format(H, H_max, int(R)))

H 	= 9.68689023671754
H_max 	= 13.909986810912573
R 	= 30%


## TFIDF Function

In [67]:
def create_tfidf(token_df, bag=WORKS, count_method='n', tf_method='sum',
                 idf_method='standard', tf_norm_k=0.5):
    """
    bag = WORKS/PARAS
    OHCO = ['work_id','part_num','chap_num','para_num','sent_num','token_num']
    count_method = 'n' # 'c' or 'n' # n = n tokens, c = distinct token (term) count
    tf_method = 'sum' # sum, max, log, double_norm, raw, binary
    tf_norm_k = .5 # only used for double_norm
    idf_method = 'standard' # standard, max, smooth
    """

    # Create Bag of Words
    BOW = token_df.groupby(bag + ['term_id']).term_id.count().to_frame().rename(columns={'term_id':'n'})
    BOW['c'] = BOW.n.astype('bool').astype('int')

    # Create Document-Term Count Matrix
    DTCM = BOW[count_method].unstack().fillna(0).astype('int')

    # Compute Term Frequency
    if tf_method == 'sum':
        TF = DTCM.T / DTCM.T.sum()
    elif tf_method == 'max':
        TF = DTCM.T / DTCM.T.max()
    elif tf_method == 'log':
        TF = np.log10(1 + DTCM.T)
    elif tf_method == 'raw':
        TF = DTCM.T
    elif tf_method == 'double_norm':
        TF = DTCM.T / DTCM.T.max()
        TF = tf_norm_k + (1 - tf_norm_k) * TF[TF > 0]
    elif tf_method == 'binary':
        TF = DTCM.T.astype('bool').astype('int')
    else:
        raise ValueError(f"Invalid tf_method: '{tf_method}'")
    
    TF = TF.T

    # Compute DF
    DF = DTCM[DTCM > 0].count()

    # Compute IDF
    N = DTCM.shape[0]
    if idf_method == 'standard':
        IDF = np.log10(N / DF)
    elif idf_method == 'max':
        IDF = np.log10(DF.max() / DF)
    elif idf_method == 'smooth':
        IDF = np.log10((N + 1) / (DF + 1)) + 1
    else:
        raise ValueError(f"Invalid idf_method: '{idf_method}'")

    # Compute TFIDF
    TFIDF = TF * IDF

    return {'BOW': BOW, 
            'DTCM': DTCM, 
            'TF': TF, 
            'DF': DF, 
            'IDF': IDF, 
            'TFIDF': TFIDF
    }

## Work-level

In [68]:
work_results = create_tfidf(TOKEN, bag = WORKS, count_method = 'n', tf_method='sum', idf_method = 'standard')

In [69]:
BOW_WORK = work_results["BOW"]
TFIDF_WORK = work_results["TFIDF"]
DF_WORK = work_results["DF"]
IDF_WORK = work_results['IDF']

In [70]:
## add tf and tfidf to BOW
BOW_WORK['tf'] = work_results['TF'].stack()
BOW_WORK['tfidf'] = TFIDF_WORK.stack()

print(f"BOW_WORK: {len(BOW_WORK):,} (doc, term) rows")
BOW_WORK.head()

BOW_WORK: 53,459 (doc, term) rows


n  c        tf     tfidf
work_id                       term_id                           
accomplishment_of_predictions 85        1  1  0.000932  0.001131
                              123       2  1  0.001864  0.002589
                              138       1  1  0.000932  0.001131
                              202      17  1  0.015843  0.000000
                              223       1  1  0.000932  0.000306

In [71]:
## add work level corpus stats to VOCAB
VOCAB['df'] = DF_WORK
VOCAB['idf'] = IDF_WORK

## per term TFIDF aggregates
VOCAB['tfidf_sum'] = TFIDF_WORK.sum()
VOCAB['tfidf_mean'] = TFIDF_WORK[TFIDF_WORK > 0].mean().fillna(0)
VOCAB['tfidf_max'] = TFIDF_WORK.max()

VOCAB.head()

,term_rank,term_str,n,num,stop,p_stem,pos_max,wn_pos,lemma,p,zipf_k,h,df,idf,tfidf_sum,tfidf_mean,tfidf_max
term_id,,,,,,,,,,,,,,,,,
13682,1,the,14903,0,1,the,DT,n,the,0.052483,14903,0.223157,49,0.000000,0.000000,0.000000,0.000000
9358,2,of,10196,0,1,of,IN,n,of,0.035906,20392,0.172337,49,0.000000,0.000000,0.000000,0.000000
726,3,and,9565,0,1,and,CC,n,and,0.033684,28695,0.164776,49,0.000000,0.000000,0.000000,0.000000
13874,4,to,9088,0,1,to,TO,n,to,0.032005,36352,0.158921,48,0.008955,0.014051,0.000293,0.000412
202,5,a,6707,0,1,a,DT,n,a,0.023620,33535,0.127637,49,0.000000,0.000000,0.000000,0.000000


In [72]:
## highest TFIDFs
VOCAB.sort_values('tfidf_sum',ascending=False).head(20)[['term_str','n','pos_max','term_rank','df','idf','tfidf_sum']]

,term_str,n,pos_max,term_rank,df,idf,tfidf_sum
term_id,,,,,,,
13824,thy,60,NNP,498,10,0.690196,0.087182
8729,miss,589,NNP,62,5,0.991226,0.081802
6828,i,5498,PRP,6,39,0.099131,0.058128
6498,her,666,PRP$,54,33,0.171682,0.054348
9093,neverout,391,JJ,92,4,1.088136,0.052392
15360,you,1347,PRP,31,34,0.158717,0.045513
13014,stella,19,NNP,1504,6,0.912045,0.043144
7746,lady,429,JJ,80,13,0.576253,0.039200
15368,your,621,PRP$,57,30,0.213075,0.039045


In [73]:
## highest TFIDFS excluding NNPs
VOCAB.loc[VOCAB.pos_max != 'NNP'].sort_values('tfidf_sum', ascending = False).head(20)[['term_str','n','pos_max','term_rank','df','idf','tfidf_sum']]

,term_str,n,pos_max,term_rank,df,idf,tfidf_sum
term_id,,,,,,,
6828,i,5498,PRP,6,39,0.099131,0.058128
6498,her,666,PRP$,54,33,0.171682,0.054348
9093,neverout,391,JJ,92,4,1.088136,0.052392
15360,you,1347,PRP,31,34,0.158717,0.045513
7746,lady,429,JJ,80,13,0.576253,0.039200
15368,your,621,PRP$,57,30,0.213075,0.039045
12256,she,400,PRP,86,25,0.292256,0.031225
2560,col,203,JJ,156,3,1.213075,0.030919
8967,my,2515,PRP$,12,37,0.121994,0.028558


## Paragraph-level

In [74]:
# paragraph-level uses long format computation approach for tractability. 
# work-level used dense-matrix approach because its smaller.

BOW_PARA = TOKEN.groupby(PARAS + ['term_id']).term_id.count()\
    .to_frame().rename(columns={'term_id':'n'})

doc_lengths = BOW_PARA.groupby(PARAS).n.sum()
BOW_PARA['tf'] = BOW_PARA.n / doc_lengths

DF_PARA = BOW_PARA.groupby('term_id').size()
N_para = doc_lengths.shape[0]
IDF_PARA = np.log10(N_para / DF_PARA)

BOW_PARA = BOW_PARA.join(IDF_PARA.rename('idf'), on='term_id')
BOW_PARA['tfidf'] = BOW_PARA.tf * BOW_PARA.idf

print(f"BOW_PARA: {len(BOW_PARA):,} (doc, term) rows")
BOW_PARA.head()

BOW_PARA: 188,023 (doc, term) rows


n        tf  \
work_id                       part_num chap_num para_num term_id                
accomplishment_of_predictions 1        1        1        123      1  0.071429   
                                                         302      1  0.071429   
                                                         630      1  0.071429   
                                                         710      1  0.071429   
                                                         3475     1  0.071429   

                                                                       idf  \
work_id                       part_num chap_num para_num term_id             
accomplishment_of_predictions 1        1        1        123      2.951702   
                                                         302      1.463857   
                                                         630      2.951702   
                                                         710      0.765593   
                                                         3475     1.667271   

                                                                     tfidf  
work_id                       part_num chap_num para_num term_id            
accomplishment_of_predictions 1        1        1        123      0.210836  
                                                         302      0.104561  
                                                         630      0.210836  
                                                         710      0.054685  
                                                         3475     0.119091

In [75]:
TOKEN.to_csv(f"{F4_path}/TOKEN.csv")
VOCAB.to_csv(f"{F4_path}/VOCAB.csv")
BOW_WORK = BOW_WORK[['n','tf','tfidf']]
BOW_WORK.to_csv(f"{F4_path}/BOW_WORK.csv")
BOW_PARA = BOW_PARA[['n','tf','tfidf']]
BOW_PARA.to_csv(f"{F4_path}/BOW_PARA.csv")